# Your first neural network

**Foundations of AI, Saturday 26 September 2026. The project.**

This morning you priced houses by hand: a guess, a miss, a correction, and
again. Now you build a machine that does it, and by the end of this page it
will put a price on houses it has never seen.

About an hour, reading included. No software, no installation, no experience.
Almost all of the code is written for you.

Everywhere you see 🎯 it is your turn. There are five of them.
Everything else runs by itself.

## Before you start: how this page works

This is a **Colab notebook**: a web page made of two kinds of block. Text, like
this one, and small blocks of **code** that you run.

**To run a block of code**, move your mouse over it. A round play button appears
on its left. Click it. You can also press **Shift and Enter** together.

**Run them in order, from the top.** Each block uses what the ones above it have
already done, so skipping one produces an error further down.

**While a block runs** it shows a spinning circle. When it has finished, the
circle turns into a small number. Most blocks here finish instantly, and the
slowest takes about two seconds.

**The first time you run anything**, Google warns you that this notebook was not
written by Google. Click **Run anyway**. You also have to be signed in to a
Google account.

**To keep your changes**, use *File, Save a copy in Drive* before you start.
Otherwise everything disappears when you close the tab, which is fine for this.

**A red box is normal.** If you run a block before filling in its blank, or fill
it in wrongly, the block turns red. That is not a disaster and nothing is
broken. Read the **last line** of the red box, the one written in plain words,
fix the blank and run the block again, as often as you like.

**If you get really stuck**, use *Runtime, Restart session* and run the blocks
again from the top. Nothing here can break anything, on your computer or
anywhere else.

A block that appears as a grey bar with a title is one you never need to read.
It sets something up. Run it and move on. The first one is below.

In [ ]:
# @title Run this cell first (it builds the street and the checkers) { display-mode: "form" }

import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)

# The street is INVENTED. 200 houses made up by the four lines below, so that
# nobody's real address is in this notebook and so that we know the answer the
# network is trying to find.
N = 200
size_m2 = 40.0 + 140.0 * torch.rand(N)
rooms = torch.round(1.0 + 4.0 * torch.rand(N))
age_years = torch.round(80.0 * torch.rand(N))
km_to_station = 0.2 + 7.8 * torch.rand(N)

price_chf = (8500.0 * size_m2 + 25000.0 * rooms - 1200.0 * age_years
             - 35000.0 * km_to_station + 200000.0
             + 60000.0 * torch.randn(N))

FEATURES = ["size in m2", "rooms", "age in years", "km to the station"]
raw = torch.stack([size_m2, rooms, age_years, km_to_station], dim=1)

cut = 160
raw_train, raw_test = raw[:cut], raw[cut:]
price_train, price_test = price_chf[:cut], price_chf[cut:]

# The four columns are on wildly different scales, so each one is put on the
# same footing before it reaches the network. The averages come from the
# training houses only, because the test houses are meant to be unseen.
mean, std = raw_train.mean(0), raw_train.std(0)
X_train = (raw_train - mean) / std
X_test = (raw_test - mean) / std

# The network learns prices in units of CHF 100,000, which keeps its numbers
# small. to_francs turns them back into money for everything you read.
SCALE = 100000.0
y_train = (price_train / SCALE).unsqueeze(1)
y_test = (price_test / SCALE).unsqueeze(1)
actual_francs = price_test


def to_francs(prediction):
    return prediction.squeeze(1) * SCALE


MODEL_FOR_CHECK = [None]
OPTIMIZER_FOR_CHECK = [None]

_ALPHABET = "ABCDEFGHJKMNPQRSTUVWXYZ"
_SALT = "FAI-HS26-HOUSES"
_NQ = 5
_first = {}


def _fnv1a32(text):
    h = 0x811C9DC5
    for b in text.encode("ascii"):
        h ^= b
        h = (h * 0x01000193) & 0xFFFFFFFF
    return h


def _seed():
    if "seed" not in _first:
        import random
        r = random.SystemRandom()
        _first["seed"] = "".join(r.choice(_ALPHABET) for _ in range(4))
    return _first["seed"]


def completion_code():
    seed = _seed()
    bits = 0
    for i in range(_NQ):
        if _first.get(i + 1) is True:
            bits |= 1 << i
    tag = bits ^ (_fnv1a32(_SALT + "|" + seed) % (1 << _NQ))
    out = ""
    v = tag
    for _ in range(3):
        out = _ALPHABET[v % 23] + out
        v //= 23
    chk = _ALPHABET[_fnv1a32(_SALT + "|" + seed + "|" + out) % 23]
    return seed + out + chk


def _record(n, ok):
    if n not in _first:
        _first[n] = ok


def _say(ok, n, good, bad):
    _record(n, ok)
    if ok:
        print("Right. " + good)
    else:
        print("Not yet. " + bad)
    return ok


def build_network(measurements, hidden, bend, answers):
    if bend is Ellipsis:
        raise ValueError("Bullseye 1a is still blank. The bend goes there: write nn.ReLU()")
    if not isinstance(bend, nn.Module):
        raise ValueError("That is not a bend. Write nn.ReLU(), with the brackets.")
    if answers is Ellipsis:
        raise ValueError("Bullseye 1b is still blank. How many numbers is a price?")
    if not isinstance(answers, int) or isinstance(answers, bool) or answers < 1:
        raise ValueError("That has to be a whole number. A price is one number.")
    return nn.Sequential(nn.Linear(measurements, hidden), bend, nn.Linear(hidden, answers))


def _watch(opt):
    if getattr(opt, "_calls", None) is not None:
        return
    opt._calls = []
    real_zero = opt.zero_grad
    real_step = opt.step

    def zero_grad(*a, **k):
        opt._calls.append("clear")
        return real_zero(*a, **k)

    def step(*a, **k):
        opt._calls.append("step")
        return real_step(*a, **k)

    opt.zero_grad = zero_grad
    opt.step = step


def check_1(net):
    layers = list(net)
    linear = [m for m in layers if isinstance(m, nn.Linear)]
    bends = [m for m in layers if isinstance(m, (nn.ReLU, nn.Tanh, nn.Sigmoid))]
    if not bends:
        return _say(False, 1, "",
                    "There is no bend between the two layers. Without one, two layers do "
                    "exactly what one layer does, and the network can only ever draw a "
                    "straight line through the street. Write nn.ReLU() on that line.")
    if len(linear) < 2 or linear[-1].out_features != 1:
        got = linear[-1].out_features if linear else 0
        return _say(False, 1, "",
                    "The last layer gives %d numbers. A price is one number, so it has to "
                    "give exactly one." % got)
    out = net(X_train[:1])
    if tuple(out.shape) != (1, 1):
        return _say(False, 1, "",
                    "The network answers with the wrong shape: %s." % (tuple(out.shape),))
    return _say(True, 1,
                "Four numbers about a house go in, the first layer turns them into 8, the "
                "bend throws away the negative ones, and the last layer turns those into "
                "one price.", "")


def check_2(opt, net):
    given = []
    for group in opt.param_groups:
        given.extend(group["params"])
    wanted = list(net.parameters())
    if not given:
        return _say(False, 2, "",
                    "The optimizer has been given nothing to turn. It needs the list of "
                    "the network's knobs, which is model.parameters()")
    if len(given) != len(wanted) or any(a is not b for a, b in zip(given, wanted)):
        return _say(False, 2, "",
                    "Those are not this network's knobs. Pass model.parameters(), with the "
                    "brackets, so the optimizer turns the layers you just built.")
    _watch(opt)
    OPTIMIZER_FOR_CHECK[0] = opt
    n = sum(p.numel() for p in wanted)
    return _say(True, 2,
                "The optimizer now has all %d knobs of your network and may turn every one "
                "of them." % n, "")


def check_3(history, epochs):
    if len(history) == 0:
        return _say(False, 3, "",
                    "The loop body never ran, so nothing was written down. The loop has to "
                    "go over range(EPOCHS).")
    if len(history) != epochs:
        return _say(False, 3, "",
                    "The loop ran %d times and EPOCHS is %d. Use range(EPOCHS)."
                    % (len(history), epochs))
    return _say(True, 3,
                "%d passes over the street, with a loss written down after every one."
                % epochs, "")


def check_4(history, net):
    if len(history) < 2:
        return _say(False, 4, "", "Nothing was trained, so there is nothing to check yet.")
    calls = getattr(OPTIMIZER_FOR_CHECK[0], "_calls", None) or []
    tail = calls[-2 * len(history):]
    if tail.count("clear") == 0:
        return _say(False, 4, "",
                    "The old tilts are never cleared. PyTorch adds every new tilt on top of "
                    "the ones already there, so after a few hundred passes it is walking on "
                    "a sum of everything it ever measured. Line 4a is optimizer.zero_grad()")
    if tail.count("step") == 0:
        return _say(False, 4, "",
                    "Nothing ever takes a step, so no knob was turned. Line 4c is "
                    "optimizer.step()")
    if tail and tail[0] != "clear":
        return _say(False, 4, "",
                    "The step is being taken before the tilts are cleared. Clearing comes "
                    "first, then working out the new tilts, then the step: you cannot walk "
                    "downhill before you have looked at the ground.")
    with torch.no_grad():
        miss = (to_francs(net(X_train)) - price_train).abs().mean().item()
    if history[-1] > 0.6 or miss > 120000:
        return _say(False, 4, "",
                    "The loss started at %.2f and ended at %.2f, and on the houses it "
                    "learned from the network is still typically wrong by CHF %s. The three "
                    "lines are in the wrong order."
                    % (history[0], history[-1], format(int(miss), ",d")))
    return _say(True, 4,
                "The loss fell from %.2f to %.2f. That is guess, check, correct, over and "
                "over, and nobody told the network what a square metre is worth."
                % (history[0], history[-1]), "")


def check_5(miss):
    if not torch.is_tensor(miss):
        try:
            miss = torch.tensor(float(miss))
        except Exception:
            return _say(False, 5, "", "That is not a number.")
    if miss.numel() != 1:
        return _say(False, 5, "",
                    "That is %d numbers, not one. You want the average of all of them, so "
                    "finish with .mean()" % miss.numel())
    value = float(miss)
    with torch.no_grad():
        want = (to_francs(MODEL_FOR_CHECK[0](X_test)) - actual_francs).abs().mean().item()
    if value < 0:
        return _say(False, 5, "",
                    "A miss cannot be negative. Some houses are guessed too high and some "
                    "too low, and those cancel out unless you make each one positive first "
                    "with .abs()")
    if abs(value - want) > max(1.0, 0.02 * want):
        return _say(False, 5, "",
                    "That comes to CHF %s and it should be about CHF %s. Take the "
                    "difference between the two lists, make it positive with .abs(), then "
                    "average it with .mean()"
                    % (format(int(value), ",d"), format(int(want), ",d")))
    return _say(True, 5,
                "Typically wrong by about CHF %s on houses it had never seen, against an "
                "average price of CHF %s. That is the only number worth quoting."
                % (format(int(want), ",d"), format(int(actual_francs.mean()), ",d")), "")


def show_street():
    print("200 invented houses. Four numbers each.")
    print()
    print("   %-12s %-8s %-14s %-18s price" % tuple(FEATURES))
    for i in [0, 1, 2, 160, 161]:
        print("   %-12.0f %-8.0f %-14.0f %-18.1f CHF %s"
              % (raw[i][0], raw[i][1], raw[i][2], raw[i][3],
                 format(int(price_chf[i]), ",d")))
    print()
    print("%d houses to learn from, %d kept back to test on."
          % (X_train.shape[0], X_test.shape[0]))
    print("Average price on the street: CHF %s" % format(int(price_chf.mean()), ",d"))
    print()
    print("The street is invented, so the prices really were made from the four")
    print("numbers plus some noise. Nothing here is anybody's real house.")


def plot_loss(history):
    plt.figure(figsize=(7, 3.2))
    plt.plot(history, color="#0072B2")
    plt.xlabel("pass over the street")
    plt.ylabel("how wrong the network is")
    plt.title("The loss, as training goes on")
    plt.grid(True, alpha=0.3)
    plt.show()


def one_house(net):
    with torch.no_grad():
        guess = to_francs(net(X_test[:1]))[0].item()
    print("One house the network never saw:")
    for name, value in zip(FEATURES, raw_test[0].tolist()):
        print("   %-20s %.1f" % (name, value))
    print()
    print("   the network says   CHF %s" % format(int(guess), ",d"))
    print("   it actually sold for   CHF %s" % format(int(actual_francs[0]), ",d"))
    print("   out by   CHF %s" % format(int(abs(guess - actual_francs[0])), ",d"))


def train_again(rate, passes):
    torch.manual_seed(0)
    net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
    fn = nn.MSELoss()
    opt = torch.optim.Adam(net.parameters(), lr=rate)
    history = []
    for _ in range(passes):
        loss = fn(net(X_train), y_train)
        opt.zero_grad()
        loss.backward()
        opt.step()
        history.append(loss.item())
    with torch.no_grad():
        miss = (to_francs(net(X_test)) - actual_francs).abs().mean().item()
    print("step size %-10s   loss at the end %8.3f   typically out by CHF %s"
          % (rate, history[-1], format(int(miss), ",d")))
    plot_loss(history)


print("Ready. 200 invented houses, nothing installed, nothing downloaded.")


## 1. The street

Nothing to fill in. Run the block and look at what we have.

In [ ]:
show_street()

Each house is **four numbers**, and each one sold for a price. The
network's job is to turn those four numbers into the right price.

Notice the last lines. Forty houses are put in a drawer and the network is never
shown them while it learns. Those forty are the only honest test of whether it
learned anything, which is the point that kept coming back today.

## 2. The network

This morning you saw what a neuron is: a **weighted sum, and then a bend**.
Stack those into layers and you have a neural network.

Ours is about as small as a network gets:

```
   4 numbers about a house  ->  8 hidden numbers  ->  bend  ->  1 price
```

🎯 **1. Two blanks.**

**a)** The bend. Without it, two layers do exactly what one layer does, and the
network could only draw a straight line through the street. In PyTorch the bend
is `nn.ReLU()`, and it does one thing: every negative number becomes zero.

**b)** How many numbers come out at the end. A price is how many numbers?

In [ ]:
model = build_network(
    measurements = 4,   # four numbers per house go in
    hidden = 8,         # eight numbers in the middle
    bend = ...,         # 🎯 1a) the bend goes here
    answers = ...,      # 🎯 1b) how many numbers is a price
)

print(model)
MODEL_FOR_CHECK[0] = model
check_1(model)

## 3. What to measure, and what to turn

Two things before training.

The **loss** is the single number saying how wrong the network is right now.
Here it is the squared miss, averaged over the houses: guess a price, see how
far off it was, square it so that a miss in either direction counts, average.
That is `nn.MSELoss()` and it is written for you.

The **optimizer** is what actually turns the knobs. It needs two things: which
knobs it may turn, and how big a step to take.

🎯 **2.** Tell the optimizer which knobs it may turn. Any
network in PyTorch will list its own if you ask it: `model.parameters()`

In [ ]:
loss_fn = nn.MSELoss()

optimizer = torch.optim.Adam(
    ...,        # 🎯 2) which knobs the optimizer may turn
    lr=0.05,    # the step size, from this morning
)

check_2(optimizer, model)

`lr` is the **step size**: the very number that made the machine bounce for
ever this morning when it was too big. Section 7 lets you play with it.

## 4. Training

This is the loop from the whole day. Once per pass over the street:

1. **guess** every price,
2. **check** how wrong that was, which is the loss,
3. **correct** every knob a little, against the tilt.

🎯 **3.** The loop itself. The network goes over all the houses
`EPOCHS` times. Use `range(EPOCHS)`.

🎯 **4.** The three lines that make PyTorch learn. Use each of
these **once**, and work out which goes where:

```
optimizer.zero_grad()      loss.backward()      optimizer.step()
```

- **a)** clear the tilts left over from the last pass, because PyTorch adds new
  ones on top of whatever is already there
- **b)** work out, for every knob, which way makes this loss smaller
- **c)** nudge every knob one step in that direction

Think about which has to happen before which. Getting it wrong breaks nothing,
and the checker will tell you what went where.

In [ ]:
EPOCHS = 400
loss_history = []

for epoch in ...:    # 🎯 3) go over the street EPOCHS times

    guesses = model(X_train)              # the guess
    loss = loss_fn(guesses, y_train)      # how wrong the guess was

    ...    # 🎯 4a) clear the old tilts
    ...    # 🎯 4b) work out the new tilts
    ...    # 🎯 4c) take one step

    loss_history.append(loss.item())

check_3(loss_history, EPOCHS)
check_4(loss_history, model)

### The learning curve

Nothing to fill in. This is the picture of the loss falling, the one behind
every training run you will ever be shown.

In [ ]:
plot_loss(loss_history)

## 5. The two numbers

The network is now good on the houses it learned from. On its own that says
nothing, because it saw their prices while it learned.

The forty in the drawer are the honest test.

🎯 **5.** How far off is it, on average, on those forty?
`predicted_francs` and `actual_francs` are two lists of prices. Take the
difference, make every miss positive with `.abs()` so that guessing too high and
too low do not cancel out, then average with `.mean()`

In [ ]:
with torch.no_grad():                 # only looking, not learning
    predicted_francs = to_francs(model(X_test))

typical_miss = ...    # 🎯 5) the average miss, in francs

check_5(typical_miss)

That number, on houses the network never saw, is the only one you would
quote to anybody. It is the answer to *"measured on what, and had the model seen
it?"*

## 6. One house, up close

Nothing to fill in.

In [ ]:
one_house(model)

## 7. Break it on purpose

No blanks here, and this is the most useful part of the page.

The step size is the one knob that has to be right. Change the number below and
run the block. It trains a fresh network from scratch and shows what happened.

Try **0.05** first, the one you used. Then **0.00001**. Then **20**.

- too small and it crawls: the loss barely moves in four hundred passes
- about right and it settles
- too big and it never settles at all

This is the grasshopper, and it is why a retrain slips by three weeks.

In [ ]:
STEP_SIZE = 0.05  # @param {type:"number"}
PASSES = 400  # @param {type:"integer"}

train_again(STEP_SIZE, PASSES)

## 8. Done

You built a neural network, trained it on a street, and tested it on houses it
had never seen. Every idea in it was on the slides today:

| What you wrote | Where you saw it |
|---|---|
| the bend | a neuron is a weighted sum, then a bend |
| `model.parameters()` | the knobs that training turns |
| `lr`, the step size | the step that has to be right |
| the loop, guess, check, correct | the whole of block 1 |
| the forty houses in the drawer | measured on what, and had it seen it |

Run the last block for your code, and send it back the way you were asked to. It
carries how you did and nothing else: no name, nothing about your computer.

In [ ]:
print(completion_code())

If you want to keep going, change the `8` in section 2 to `1`, then run
everything from there again. One hidden number cannot bend, and you can watch
the street flatten into a straight line.

Thank you for coming on a Saturday.